# 00 · Start hier

Welkom bij het IoT-practicum van de les **Klimaatbeheersing in de veehouderij**. Dit notebook controleert of je omgeving werkt en frist de begrippen op die je in de volgende notebooks nodig hebt.

**Hoe werk je met een notebook?** Een notebook bestaat uit tekstcellen (zoals deze) en codecellen. Klik in een codecel en druk op **Shift + Enter** om ze uit te voeren. De uitvoer verschijnt eronder. Voer de cellen **van boven naar beneden** uit.

> Werk je in GitHub Codespaces en vraagt VS Code om een *kernel*? Kies **Python 3.12** (of de hoogste Python-versie die wordt aangeboden).

## 1. Werkt Python en zijn de pakketten geïnstalleerd?

In [ ]:
import sys, platform
import paho.mqtt.client as mqtt
import requests, pandas as pd, matplotlib

print("Python      :", sys.version.split()[0], "op", platform.system())
print("paho-mqtt   :", mqtt.__version__ if hasattr(mqtt, "__version__") else "ok")
print("requests    :", requests.__version__)
print("pandas      :", pd.__version__)
print("matplotlib  :", matplotlib.__version__)
print("\nAlles geladen. Je kunt verder.")

## 2. Begrippen die je nodig hebt

| Begrip | Betekenis in dit practicum |
|---|---|
| **IoT-toestel / node / eindpunt** | Een sensor met een radio en een batterij, die zelfstandig metingen verstuurt. Voorbeeld: Dragino S31-LB. |
| **Verbindingstype** | De *fysieke* weg waarlangs bits reizen: wifi, ethernet, LoRaWAN, Bluetooth, NB-IoT/LTE-M. Vergelijk het met de weg (fietspad, autosnelweg, spoor). |
| **Gegevensuitwisselingsprotocol** | De *afspraak* over hoe berichten eruitzien en wie wanneer spreekt: MQTT, HTTP, CoAP, Modbus. Vergelijk het met de taal en de etiquette die je op die weg gebruikt. |
| **Gateway** | Toestel dat tussen twee verbindingstypes vertaalt, bv. van LoRaWAN naar ethernet/internet. |
| **Broker** | De centrale "postbus" van MQTT: toestellen publiceren berichten op een *topic*, andere toestellen abonneren zich op dat topic. |
| **Topic** | Een adres met schuine strepen, bv. `stal1/afdeling3/temperatuur`. |
| **Payload** | De eigenlijke inhoud van een bericht: ruwe bytes (hex), tekst of JSON. |
| **JSON** | Tekstformaat met sleutels en waarden: `{"temperature": 21.4, "humidity": 63.2}`. |
| **Uplink / downlink** | Bericht van sensor naar server / van server naar sensor. |

Een vaste **rode draad** doorheen het practicum: een temperatuur- en vochtsensor in een stalafdeling verstuurt elke 10 minuten een meting, die uiteindelijk op het scherm van de veehouder of in de klimaatcomputer belandt.

## 3. Verbinding met de MQTT-broker testen

We gebruiken de **publieke broker van HiveMQ** (`broker.hivemq.com`). Iedereen ter wereld mag ermee verbinden, dus gebruik ze nooit voor echte of persoonlijke gegevens. Onderstaande cel verbindt, abonneert zich op een testtopic, publiceert één bericht en wacht tot ze dat eigen bericht terugkrijgt.

In [ ]:
import paho.mqtt.client as mqtt
import time, uuid

BROKER = "broker.hivemq.com"
POORT  = 1883                       # standaard MQTT-poort (onversleuteld)
TOPIC  = f"ugent/i700301/test/{uuid.uuid4().hex[:8]}"   # uniek testtopic

ontvangen = []

def bij_bericht(client, userdata, msg):
    ontvangen.append(msg.payload.decode())

client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
client.on_message = bij_bericht
client.connect(BROKER, POORT, keepalive=60)
client.loop_start()
client.subscribe(TOPIC)
time.sleep(1)
client.publish(TOPIC, "hallo vanuit mijn notebook")

for _ in range(50):                  # max. 5 s wachten
    if ontvangen:
        break
    time.sleep(0.1)

client.loop_stop(); client.disconnect()
print("Topic   :", TOPIC)
print("Ontvangen:", ontvangen or "NIETS — zie de tip hieronder")

**Lukt het niet (geen bericht of een time-out)?** Sommige netwerken blokkeren poort 1883. Gebruik dan MQTT **over websockets** op poort 8884 met TLS; dat lijkt voor het netwerk op gewoon https-verkeer. Vervang in de cel hierboven de regel `client = mqtt.Client(...)` en `client.connect(...)` door:

```python
client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, transport="websockets")
client.tls_set()
client.ws_set_options(path="/mqtt")
client.connect("broker.hivemq.com", 8884, keepalive=60)
```

In GitHub Codespaces werkt poort 1883 normaal gezien gewoon.

## 4. Verbinding met een HTTP-API testen

HTTP is het protocol van het web. Een *API* is een webadres dat geen webpagina maar gegevens (meestal JSON) teruggeeft. We vragen het actuele weer in Gent op bij Open-Meteo.

In [ ]:
import requests

url = "https://api.open-meteo.com/v1/forecast"
parameters = {
    "latitude": 51.05, "longitude": 3.72,          # Gent
    "current": "temperature_2m,relative_humidity_2m",
}
antwoord = requests.get(url, params=parameters, timeout=10)
print("HTTP-statuscode:", antwoord.status_code, "(200 = OK)")

gegevens = antwoord.json()                          # JSON-tekst -> Python-dictionary
huidig = gegevens["current"]
print(f"Gent, {huidig['time']} UTC: {huidig['temperature_2m']} °C, {huidig['relative_humidity_2m']} % RV")

## 5. Klaar

Werkten beide tests? Ga dan naar `01_sensor_kiezen_en_verbinden.ipynb`.

Onthoud het verschil dat je net zag:

* Bij **MQTT** maak je één verbinding, en daarna *duwt* de broker berichten naar jou zodra ze er zijn (publish/subscribe).
* Bij **HTTP** *vraag* je telkens expliciet om gegevens en krijg je één antwoord (request/response).